## Join SB County Rainfall Data to Buffered Buildings


In [1]:
# Load packages
import os
import sys
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from scipy.spatial import cKDTree

import os
import sys

sys.path.append("../utils")

import config

In [2]:
# Load building data
buffered_buildings = os.path.join(config.data_dir, "PUZZLE_PIECES", "inspections_master.geojson")
buffered_buildings = gpd.read_file(buffered_buildings)
buffered_buildings

,inspection_id,status,year,month,Date,a_removebr,a_removebranchesfromstovepipe,accessegre,accessegress,address__1,...,water_comm,water_comments,water_sour,water_source,water_stor,water_storage_size_stored_water_on_individual_parcels_only,waterstora,windowpane,yearbuilt,geometry
0,1,Compliant,2019,6,2019-06-05,None,None,yes,None,None,...,None,None,Private Hydrant- Private Stored Water,None,Over 5000 gallons,None,yes,No Windows,2000.0,"POLYGON ((-2813.179 -371821.557, -2813.179 -37..."
1,2,Compliant,2019,5,2019-05-09,None,None,yes,None,None,...,None,None,Private Hydrant- Private Stored Water,None,None,None,yes,Single Pane,0.0,"POLYGON ((-8368.416 -370860.259, -8475.394 -37..."
2,3,Compliant,2019,5,2019-05-09,None,None,yes,None,None,...,None,None,None,None,None,None,None,Multi Pane,0.0,"POLYGON ((-8253.618 -371073.707, -8136.129 -37..."
3,4,Compliant,2019,5,2019-05-09,None,None,yes,None,None,...,None,None,None,None,None,None,no,Single Pane,0.0,"POLYGON ((-8201.866 -370989.432, -8201.866 -37..."
4,5,Compliant,2019,12,2019-12-17,None,None,yes,None,None,...,None,None,Pond,None,None,None,yes,Single Pane,2002.0,"POLYGON ((-7966.144 -370818.558, -7966.144 -37..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67575,67576,Compliant,2023,10,2023-10-31,None,None,None,Yes,None,...,None,None,None,None,None,None,None,None,NaN,"POLYGON ((-34938.314 -349478.543, -34991.669 -..."
67576,67577,Compliant,2023,11,2023-11-15,None,None,None,Yes,None,...,None,None,None,None,None,None,None,None,NaN,"POLYGON ((-36511.881 -350331.419, -36511.881 -..."
67577,67578,Compliant,2023,11,2023-11-15,None,None,None,Yes,None,...,None,None,None,None,None,None,None,None,NaN,"POLYGON ((-38702.857 -346029.921, -38702.857 -..."
67578,67579,Compliant,2023,11,2023-11-15,None,None,None,Yes,None,...,None,None,None,None,None,None,None,None,NaN,"POLYGON ((-38988.592 -346062.709, -38884.895 -..."


In [3]:
# Load rain data
rain_data = os.path.join(config.data_dir, "sb_rain_gauges", "rain_gauge_data.csv")
rain_data = pd.read_csv(rain_data)

In [4]:
# Convert rain_data to a GeoDataFrame
rain_gdf = gpd.GeoDataFrame(
    rain_data,
    geometry=gpd.points_from_xy(rain_data['long'], rain_data['lat']),
    crs=config.albers_crs
)
rain_gdf

,station_id,year,month,monthly_rain_total,lat,long,elevation,geometry
0,196.0,2017.0,1.0,7.49,34.694167,-120.131667,1000,POINT (-120.132 34.694)
1,196.0,2017.0,2.0,6.97,34.694167,-120.131667,1000,POINT (-120.132 34.694)
2,196.0,2017.0,3.0,0.81,34.694167,-120.131667,1000,POINT (-120.132 34.694)
3,196.0,2017.0,4.0,0.53,34.694167,-120.131667,1000,POINT (-120.132 34.694)
4,196.0,2017.0,5.0,0.37,34.694167,-120.131667,1000,POINT (-120.132 34.694)
...,...,...,...,...,...,...,...,...
4604,565.0,2023.0,8.0,0.51,34.570833,-119.732222,2870,POINT (-119.732 34.571)
4605,565.0,2023.0,9.0,0.02,34.570833,-119.732222,2870,POINT (-119.732 34.571)
4606,565.0,2023.0,10.0,0.01,34.570833,-119.732222,2870,POINT (-119.732 34.571)
4607,565.0,2023.0,11.0,0.44,34.570833,-119.732222,2870,POINT (-119.732 34.571)


In [5]:
# Convert buffered building to points instead of polygons
inspection_points = gpd.GeoDataFrame(
    buffered_buildings,
    geometry=gpd.points_from_xy(buffered_buildings['longitude'], buffered_buildings['latitude']),
    crs=config.albers_crs
)
inspection_points

,inspection_id,status,year,month,Date,a_removebr,a_removebranchesfromstovepipe,accessegre,accessegress,address__1,...,water_comm,water_comments,water_sour,water_source,water_stor,water_storage_size_stored_water_on_individual_parcels_only,waterstora,windowpane,yearbuilt,geometry
0,1,Compliant,2019,6,2019-06-05,None,None,yes,None,None,...,None,None,Private Hydrant- Private Stored Water,None,Over 5000 gallons,None,yes,No Windows,2000.0,POINT (-120.031 34.67)
1,2,Compliant,2019,5,2019-05-09,None,None,yes,None,None,...,None,None,Private Hydrant- Private Stored Water,None,None,None,yes,Single Pane,0.0,POINT (-120.092 34.679)
2,3,Compliant,2019,5,2019-05-09,None,None,yes,None,None,...,None,None,None,None,None,None,None,Multi Pane,0.0,POINT (-120.09 34.676)
3,4,Compliant,2019,5,2019-05-09,None,None,yes,None,None,...,None,None,None,None,None,None,no,Single Pane,0.0,POINT (-120.089 34.678)
4,5,Compliant,2019,12,2019-12-17,None,None,yes,None,None,...,None,None,Pond,None,None,None,yes,Single Pane,2002.0,POINT (-120.088 34.678)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67575,67576,Compliant,2023,10,2023-10-31,None,None,None,Yes,None,...,None,None,None,None,None,None,None,None,NaN,POINT (-120.383 34.871)
67576,67577,Compliant,2023,11,2023-11-15,None,None,None,Yes,None,...,None,None,None,None,None,None,None,None,NaN,POINT (-120.399 34.863)
67577,67578,Compliant,2023,11,2023-11-15,None,None,None,Yes,None,...,None,None,None,None,None,None,None,None,NaN,POINT (-120.424 34.901)
67578,67579,Compliant,2023,11,2023-11-15,None,None,None,Yes,None,...,None,None,None,None,None,None,None,None,NaN,POINT (-120.426 34.901)


In [6]:
# Check that the CRS match
if inspection_points.crs != rain_gdf.crs:
    inspection_points = inspection_points.to_crs(rain_gdf.crs)


In [7]:
# Extract coordinates
inspection_coords = np.array(list(inspection_points.geometry.apply(lambda geom: (geom.x, geom.y))))
rain_coords = np.array(list(rain_gdf.geometry.apply(lambda geom: (geom.x, geom.y))))

# Build KDTree and query nearest
tree = cKDTree(rain_coords)
distances, indices = tree.query(inspection_coords, k=1)

# Extract matched rain station info
matched_rain = rain_gdf.iloc[indices].reset_index(drop=True)
matched_rain = matched_rain[['station_id']].copy()
matched_rain['station_distance'] = distances

# Join with original inspection_points
closest_rain = inspection_points.reset_index(drop=True).join(matched_rain)

closest_rain

,inspection_id,status,year,month,Date,a_removebr,a_removebranchesfromstovepipe,accessegre,accessegress,address__1,...,water_sour,water_source,water_stor,water_storage_size_stored_water_on_individual_parcels_only,waterstora,windowpane,yearbuilt,geometry,station_id,station_distance
0,1,Compliant,2019,6,2019-06-05,None,None,yes,None,None,...,Private Hydrant- Private Stored Water,None,Over 5000 gallons,None,yes,No Windows,2000.0,POINT (-120.031 34.67),421.0,0.069204
1,2,Compliant,2019,5,2019-05-09,None,None,yes,None,None,...,Private Hydrant- Private Stored Water,None,None,None,yes,Single Pane,0.0,POINT (-120.092 34.679),196.0,0.042502
2,3,Compliant,2019,5,2019-05-09,None,None,yes,None,None,...,None,None,None,None,None,Multi Pane,0.0,POINT (-120.09 34.676),196.0,0.045787
3,4,Compliant,2019,5,2019-05-09,None,None,yes,None,None,...,None,None,None,None,no,Single Pane,0.0,POINT (-120.089 34.678),196.0,0.045685
4,5,Compliant,2019,12,2019-12-17,None,None,yes,None,None,...,Pond,None,None,None,yes,Single Pane,2002.0,POINT (-120.088 34.678),196.0,0.046837
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67575,67576,Compliant,2023,10,2023-10-31,None,None,None,Yes,None,...,None,None,None,None,None,None,NaN,POINT (-120.383 34.871),349.0,0.033760
67576,67577,Compliant,2023,11,2023-11-15,None,None,None,Yes,None,...,None,None,None,None,None,None,NaN,POINT (-120.399 34.863),349.0,0.043980
67577,67578,Compliant,2023,11,2023-11-15,None,None,None,Yes,None,...,None,None,None,None,None,None,NaN,POINT (-120.424 34.901),198.0,0.030967
67578,67579,Compliant,2023,11,2023-11-15,None,None,None,Yes,None,...,None,None,None,None,None,None,NaN,POINT (-120.426 34.901),198.0,0.029038


In [28]:
# Create a year and month column to merge the rain data by
closest_rain['year'] = closest_rain['Date'].dt.year
closest_rain['month'] = closest_rain['Date'].dt.month
closest_rain['month_prior'] = closest_rain['month'].apply(lambda x: 12 if x == 1 else x - 1)
# Default year_prior to current year
closest_rain['year_prior'] = closest_rain['year']

# Set year_prior where month == 1
closest_rain.loc[closest_rain['month'] == 1, 'year_prior'] = closest_rain['year'] - 1

closest_rain['month_prior_2'] = closest_rain['month'].apply(lambda x: x - 2 if x > 2 else x + 10)
# Default year_prior to current year
closest_rain['year_prior_2'] = closest_rain['year']

# Set year_prior where month == 1
closest_rain.loc[(closest_rain['month'] == 1) | (closest_rain['month'] == 2), 'year_prior_2'] = closest_rain['year'] - 1

closest_rain

,inspection_id,status,year,month,Date,a_removebr,a_removebranchesfromstovepipe,accessegre,accessegress,address__1,...,waterstora,windowpane,yearbuilt,geometry,station_id,station_distance,month_prior,year_prior,month_prior_2,year_prior_2
0,1,Compliant,2019,6,2019-06-05,None,None,yes,None,None,...,yes,No Windows,2000.0,POINT (-120.031 34.67),421.0,0.069204,5,2019,4,2019
1,2,Compliant,2019,5,2019-05-09,None,None,yes,None,None,...,yes,Single Pane,0.0,POINT (-120.092 34.679),196.0,0.042502,4,2019,3,2019
2,3,Compliant,2019,5,2019-05-09,None,None,yes,None,None,...,None,Multi Pane,0.0,POINT (-120.09 34.676),196.0,0.045787,4,2019,3,2019
3,4,Compliant,2019,5,2019-05-09,None,None,yes,None,None,...,no,Single Pane,0.0,POINT (-120.089 34.678),196.0,0.045685,4,2019,3,2019
4,5,Compliant,2019,12,2019-12-17,None,None,yes,None,None,...,yes,Single Pane,2002.0,POINT (-120.088 34.678),196.0,0.046837,11,2019,10,2019
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67575,67576,Compliant,2023,10,2023-10-31,None,None,None,Yes,None,...,None,None,NaN,POINT (-120.383 34.871),349.0,0.033760,9,2023,8,2023
67576,67577,Compliant,2023,11,2023-11-15,None,None,None,Yes,None,...,None,None,NaN,POINT (-120.399 34.863),349.0,0.043980,10,2023,9,2023
67577,67578,Compliant,2023,11,2023-11-15,None,None,None,Yes,None,...,None,None,NaN,POINT (-120.424 34.901),198.0,0.030967,10,2023,9,2023
67578,67579,Compliant,2023,11,2023-11-15,None,None,None,Yes,None,...,None,None,NaN,POINT (-120.426 34.901),198.0,0.029038,10,2023,9,2023


In [36]:
closest_rain_merged = closest_rain.merge(
    rain_data,
    on=['station_id', 'year', 'month'],
    how='left'
)
closest_rain_merged = closest_rain_merged[['inspection_id', 'status', 'year', 'month', 'Date', 'geometry', 'station_id', 'monthly_rain_total', 'year_prior', 'month_prior', 'year_prior_2', 'month_prior_2']]
closest_rain_merged

,inspection_id,status,year,month,Date,geometry,station_id,monthly_rain_total,year_prior,month_prior,year_prior_2,month_prior_2
0,1,Compliant,2019,6,2019-06-05,POINT (-120.031 34.67),421.0,0.28,2019,5,2019,4
1,2,Compliant,2019,5,2019-05-09,POINT (-120.092 34.679),196.0,1.46,2019,4,2019,3
2,3,Compliant,2019,5,2019-05-09,POINT (-120.09 34.676),196.0,1.46,2019,4,2019,3
3,4,Compliant,2019,5,2019-05-09,POINT (-120.089 34.678),196.0,1.46,2019,4,2019,3
4,5,Compliant,2019,12,2019-12-17,POINT (-120.088 34.678),196.0,5.14,2019,11,2019,10
...,...,...,...,...,...,...,...,...,...,...,...,...
67575,67576,Compliant,2023,10,2023-10-31,POINT (-120.383 34.871),349.0,0.23,2023,9,2023,8
67576,67577,Compliant,2023,11,2023-11-15,POINT (-120.399 34.863),349.0,0.44,2023,10,2023,9
67577,67578,Compliant,2023,11,2023-11-15,POINT (-120.424 34.901),198.0,0.43,2023,10,2023,9
67578,67579,Compliant,2023,11,2023-11-15,POINT (-120.426 34.901),198.0,0.43,2023,10,2023,9


In [44]:
closest_rain_final = closest_rain_merged.merge(
    rain_data,
    left_on=['station_id', 'year_prior', 'month_prior'],
    right_on=['station_id', 'year', 'month'],
    how='left'
)

closest_rain_final.rename(columns = {'monthly_rain_total_x': 'current_month_rain',
                                     'monthly_rain_total_y': 'previous_month_rain'}
)
closest_rain_final

,inspection_id,status,year_x,month_x,Date,geometry,station_id,monthly_rain_total_x,year_prior,month_prior,year_prior_2,month_prior_2,year_y,month_y,monthly_rain_total_y,lat,long,elevation
0,1,Compliant,2019,6,2019-06-05,POINT (-120.031 34.67),421.0,0.28,2019,5,2019,4,2019.0,5.0,2.18,34.734444,-120.006944,3250.0
1,2,Compliant,2019,5,2019-05-09,POINT (-120.092 34.679),196.0,1.46,2019,4,2019,3,2019.0,4.0,0.08,34.694167,-120.131667,1000.0
2,3,Compliant,2019,5,2019-05-09,POINT (-120.09 34.676),196.0,1.46,2019,4,2019,3,2019.0,4.0,0.08,34.694167,-120.131667,1000.0
3,4,Compliant,2019,5,2019-05-09,POINT (-120.089 34.678),196.0,1.46,2019,4,2019,3,2019.0,4.0,0.08,34.694167,-120.131667,1000.0
4,5,Compliant,2019,12,2019-12-17,POINT (-120.088 34.678),196.0,5.14,2019,11,2019,10,2019.0,11.0,1.67,34.694167,-120.131667,1000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67575,67576,Compliant,2023,10,2023-10-31,POINT (-120.383 34.871),349.0,0.23,2023,9,2023,8,2023.0,9.0,0.16,34.848333,-120.357778,1048.0
67576,67577,Compliant,2023,11,2023-11-15,POINT (-120.399 34.863),349.0,0.44,2023,10,2023,9,2023.0,10.0,0.23,34.848333,-120.357778,1048.0
67577,67578,Compliant,2023,11,2023-11-15,POINT (-120.424 34.901),198.0,0.43,2023,10,2023,9,2023.0,10.0,0.07,34.882222,-120.448611,280.0
67578,67579,Compliant,2023,11,2023-11-15,POINT (-120.426 34.901),198.0,0.43,2023,10,2023,9,2023.0,10.0,0.07,34.882222,-120.448611,280.0


In [ ]:
rain_final = closest_rain_final[['inspection_id', 
                                        'station_id',
                                        'current_month_rain',
                                        'previous_month_rain',
                                        'year_prior_2',
                                        'month_prior_2']]

KeyError: "['current_month_rain', 'previous_month_rain'] not in index"

In [10]:
print(rain_data.dtypes)


station_id            float64
year                  float64
month                 float64
monthly_rain_total    float64
lat                   float64
long                  float64
elevation               int64
dtype: object


In [11]:
# Read to CSV
buffered_buildings_rain_data_joined = os.path.join(config.data_dir, "sb_rain_gauges", "buffered_buildings_rain_data_joined.csv")
closest_rain.to_csv(buffered_buildings_rain_data_joined, index=False)